In [1]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from tensorflow.keras.utils import plot_model

os.makedirs("outputs/figures", exist_ok=True)

C:\Users\Yahya\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\Yahya\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\Yahya\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please 

figure 10

In [2]:
LOOKBACK = 60
N_FEATURES = 35

def build_lstm_model(n_features, lookback, lr=1e-3, lstm_units=64, dropout=0.2):
    inp = layers.Input(shape=(lookback, n_features))
    x = layers.LSTM(lstm_units, return_sequences=False)(inp)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dense(32, activation="relu")(x)
    out = layers.Dense(1, name="y_hat")(x)

    model = keras.Model(inputs=inp, outputs=out)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss="mse")
    return model

In [3]:
! pip install graphviz


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
single_model = build_lstm_model(n_features=N_FEATURES, lookback=LOOKBACK)

plot_model(
    single_model,
    to_file="outputs/figures/Figure10_single_output_lstm_arch.png",
    show_shapes=True,
    show_layer_names=True,
    dpi=200
)

print("Saved: outputs/figures/Figure10_single_output_lstm_arch.png")

Saved: outputs/figures/Figure10_single_output_lstm_arch.png


figure 11

In [5]:
from tensorflow.keras import regularizers

def build_multi_output_lstm_reg(
    n_features,
    lookback,
    n_outputs=5,
    lr=1e-3,
    lstm_units=32,
    dropout=0.4,
    l2_lambda=1e-4
):
    inp = layers.Input(shape=(lookback, n_features))

    x = layers.LSTM(
        lstm_units,
        return_sequences=False,
        kernel_regularizer=regularizers.l2(l2_lambda),
        recurrent_regularizer=regularizers.l2(l2_lambda),
        bias_regularizer=regularizers.l2(l2_lambda)
    )(inp)

    x = layers.Dropout(dropout)(x)

    x = layers.Dense(64, activation="relu",
                     kernel_regularizer=regularizers.l2(l2_lambda),
                     bias_regularizer=regularizers.l2(l2_lambda))(x)

    x = layers.Dense(32, activation="relu",
                     kernel_regularizer=regularizers.l2(l2_lambda),
                     bias_regularizer=regularizers.l2(l2_lambda))(x)

    out = layers.Dense(n_outputs, name="y_hat")(x)

    model = keras.Model(inputs=inp, outputs=out)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss="mse")
    return model


In [6]:
multi_model_reg_arch = build_multi_output_lstm_reg(
    n_features=N_FEATURES,
    lookback=LOOKBACK,
    n_outputs=5
)

plot_model(
    multi_model_reg_arch,
    to_file="outputs/figures/Figure11_multi_output_lstm_arch.png",
    show_shapes=True,
    show_layer_names=True,
    dpi=200
)

print("Saved: outputs/figures/Figure11_multi_output_lstm_arch.png")


Saved: outputs/figures/Figure11_multi_output_lstm_arch.png
